# **Predicting High-Cost Medicare Beneficiaries: A Machine Learning Comparison Using CMS Synthetic Claims Data**

Medicare covers 65M+ beneficiaries and exceeds $100B in annual Part D spending, with a small subset of patients driving a disproportionate share of total costs. Early identification of these high-cost beneficiaries enables targeted care management, supports value-based care contracts, and informs specialty drug program design.

This project uses CMS De-SynPUF Sample 9, a 0.25% subsample containing ~115k beneficiaries per year (2008–2010) and millions of associated claims, including 4.7M Carrier, 66k inpatient, 791k outpatient, and 5.55M PDE records. These files span inpatient, outpatient, professional, and pharmacy utilization to support comprehensive feature development.

Although the dataset is synthetic and based on ICD-9–era patterns, its structure closely mirrors real CMS claims, making it useful for methodological testing. This project will compare different ML algorithm performances on high-cost beneficiaries predictions and evaluate such outputs through AUC, calibration, and other relevant metrics.

In [1]:
import os
import random
import pandas as pd
import numpy as np
import sklearn
import xgboost as xgb
from platform import python_version
from src.cleaning import *

seed = 5499
random.seed(seed)
np.random.seed(seed)

print("Python version used:", python_version())
print("Sklearn version used:", sklearn.__version__)
print("XGBoost version used:", xgb.__version__)

Python version used: 3.13.5
Sklearn version used: 1.6.1
XGBoost version used: 3.1.2


In [2]:
beneficiaries = pd.read_pickle("data/cache/beneficiary.pkl")
carrier = pd.read_pickle("data/cache/carrier.pkl")
inpatient = pd.read_pickle("data/cache/inpatient.pkl")
outpatient = pd.read_pickle("data/cache/outpatient.pkl")
pde = pd.read_pickle("data/cache/pde.pkl")

In [ ]:
audit_df(beneficiaries, "beneficiaries", "DESYNPUF_ID")

In [ ]:
death_records = beneficiaries[beneficiaries["BENE_DEATH_DT"].notna()]
death_records.groupby("year").count()

In [ ]:
excluded_cohort = death_records["DESYNPUF_ID"].unique()
beneficiaries_cohort = beneficiaries[~beneficiaries["DESYNPUF_ID"].isin(excluded_cohort)]
beneficiaries_cohort.shape

In [ ]:
beneficiaries_cohort["DESYNPUF_ID"].nunique() * 3 == beneficiaries_cohort.shape[0]

In [ ]:
refined_cohort1 = mapping_beneficiary_demographics(beneficiaries_cohort)
refined_cohort2 = refined_cohort1.drop("BENE_DEATH_DT", axis=1)
refined_cohort2.head()

In [ ]:
refined_cohort = recoding_beneficiary_flags(refined_cohort2)
refined_cohort.head()

In [ ]:
audit_df(carrier, "carrier", "DESYNPUF_ID")

In [ ]:
carrier["CLM_ID"].nunique()

In [ ]:
carrier[carrier["CLM_THRU_DT"] < carrier["CLM_FROM_DT"]]

In [4]:
# check the logics behind M to confirm the duplication is another line within the same claim, and hence needing to turn the corresponding payments for lines with M into 0
relevant_cols = (
    ["CLM_ID"]
    + [f"LINE_PRCSG_IND_CD_{i}" for i in range(1, 14)]
    + [f"LINE_NCH_PMT_AMT_{i}" for i in range(1, 14)]
    + [f"LINE_ALOWD_CHRG_AMT_{i}" for i in range(1, 14)]
    + [f"LINE_COINSRNC_AMT_{i}" for i in range(1, 14)]
)
has_M = carrier[relevant_cols].isin(["M"]).any(axis=1)
carrier_sample = carrier[has_M].sample(n=5, random_state=42)
carrier_sample[relevant_cols]
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [5]:
carrier_sample.head()

,DESYNPUF_ID,CLM_ID,CLM_FROM_DT,CLM_THRU_DT,ICD9_DGNS_CD_1,ICD9_DGNS_CD_2,ICD9_DGNS_CD_3,ICD9_DGNS_CD_4,ICD9_DGNS_CD_5,ICD9_DGNS_CD_6,ICD9_DGNS_CD_7,ICD9_DGNS_CD_8,PRF_PHYSN_NPI_1,PRF_PHYSN_NPI_2,PRF_PHYSN_NPI_3,PRF_PHYSN_NPI_4,PRF_PHYSN_NPI_5,PRF_PHYSN_NPI_6,PRF_PHYSN_NPI_7,PRF_PHYSN_NPI_8,PRF_PHYSN_NPI_9,PRF_PHYSN_NPI_10,PRF_PHYSN_NPI_11,PRF_PHYSN_NPI_12,PRF_PHYSN_NPI_13,TAX_NUM_1,TAX_NUM_2,TAX_NUM_3,TAX_NUM_4,TAX_NUM_5,TAX_NUM_6,TAX_NUM_7,TAX_NUM_8,TAX_NUM_9,TAX_NUM_10,TAX_NUM_11,TAX_NUM_12,TAX_NUM_13,HCPCS_CD_1,HCPCS_CD_2,HCPCS_CD_3,HCPCS_CD_4,HCPCS_CD_5,HCPCS_CD_6,HCPCS_CD_7,HCPCS_CD_8,HCPCS_CD_9,HCPCS_CD_10,HCPCS_CD_11,HCPCS_CD_12,HCPCS_CD_13,LINE_NCH_PMT_AMT_1,LINE_NCH_PMT_AMT_2,LINE_NCH_PMT_AMT_3,LINE_NCH_PMT_AMT_4,LINE_NCH_PMT_AMT_5,LINE_NCH_PMT_AMT_6,LINE_NCH_PMT_AMT_7,LINE_NCH_PMT_AMT_8,LINE_NCH_PMT_AMT_9,LINE_NCH_PMT_AMT_10,LINE_NCH_PMT_AMT_11,LINE_NCH_PMT_AMT_12,LINE_NCH_PMT_AMT_13,LINE_BENE_PTB_DDCTBL_AMT_1,LINE_BENE_PTB_DDCTBL_AMT_2,LINE_BENE_PTB_DDCTBL_AMT_3,LINE_BENE_PTB_DDCTBL_AMT_4,LINE_BENE_PTB_DDCTBL_AMT_5,LINE_BENE_PTB_DDCTBL_AMT_6,LINE_BENE_PTB_DDCTBL_AMT_7,LINE_BENE_PTB_DDCTBL_AMT_8,LINE_BENE_PTB_DDCTBL_AMT_9,LINE_BENE_PTB_DDCTBL_AMT_10,LINE_BENE_PTB_DDCTBL_AMT_11,LINE_BENE_PTB_DDCTBL_AMT_12,LINE_BENE_PTB_DDCTBL_AMT_13,LINE_BENE_PRMRY_PYR_PD_AMT_1,LINE_BENE_PRMRY_PYR_PD_AMT_2,LINE_BENE_PRMRY_PYR_PD_AMT_3,LINE_BENE_PRMRY_PYR_PD_AMT_4,LINE_BENE_PRMRY_PYR_PD_AMT_5,LINE_BENE_PRMRY_PYR_PD_AMT_6,LINE_BENE_PRMRY_PYR_PD_AMT_7,LINE_BENE_PRMRY_PYR_PD_AMT_8,LINE_BENE_PRMRY_PYR_PD_AMT_9,LINE_BENE_PRMRY_PYR_PD_AMT_10,LINE_BENE_PRMRY_PYR_PD_AMT_11,LINE_BENE_PRMRY_PYR_PD_AMT_12,LINE_BENE_PRMRY_PYR_PD_AMT_13,LINE_COINSRNC_AMT_1,LINE_COINSRNC_AMT_2,LINE_COINSRNC_AMT_3,LINE_COINSRNC_AMT_4,LINE_COINSRNC_AMT_5,LINE_COINSRNC_AMT_6,LINE_COINSRNC_AMT_7,LINE_COINSRNC_AMT_8,LINE_COINSRNC_AMT_9,LINE_COINSRNC_AMT_10,LINE_COINSRNC_AMT_11,LINE_COINSRNC_AMT_12,LINE_COINSRNC_AMT_13,LINE_ALOWD_CHRG_AMT_1,LINE_ALOWD_CHRG_AMT_2,LINE_ALOWD_CHRG_AMT_3,LINE_ALOWD_CHRG_AMT_4,LINE_ALOWD_CHRG_AMT_5,LINE_ALOWD_CHRG_AMT_6,LINE_ALOWD_CHRG_AMT_7,LINE_ALOWD_CHRG_AMT_8,LINE_ALOWD_CHRG_AMT_9,LINE_ALOWD_CHRG_AMT_10,LINE_ALOWD_CHRG_AMT_11,LINE_ALOWD_CHRG_AMT_12,LINE_ALOWD_CHRG_AMT_13,LINE_PRCSG_IND_CD_1,LINE_PRCSG_IND_CD_2,LINE_PRCSG_IND_CD_3,LINE_PRCSG_IND_CD_4,LINE_PRCSG_IND_CD_5,LINE_PRCSG_IND_CD_6,LINE_PRCSG_IND_CD_7,LINE_PRCSG_IND_CD_8,LINE_PRCSG_IND_CD_9,LINE_PRCSG_IND_CD_10,LINE_PRCSG_IND_CD_11,LINE_PRCSG_IND_CD_12,LINE_PRCSG_IND_CD_13,LINE_ICD9_DGNS_CD_1,LINE_ICD9_DGNS_CD_2,LINE_ICD9_DGNS_CD_3,LINE_ICD9_DGNS_CD_4,LINE_ICD9_DGNS_CD_5,LINE_ICD9_DGNS_CD_6,LINE_ICD9_DGNS_CD_7,LINE_ICD9_DGNS_CD_8,LINE_ICD9_DGNS_CD_9,LINE_ICD9_DGNS_CD_10,LINE_ICD9_DGNS_CD_11,LINE_ICD9_DGNS_CD_12,LINE_ICD9_DGNS_CD_13
4039511,D9EE49D4A9633275,684623372337752,20080128,20080128,36275,37921,NaN,NaN,NaN,NaN,NaN,NaN,2.966587e+09,2.966587e+09,2.966587e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,258582083,258582083.0,258582083.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,99213,92015,92015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,40.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,90.0,20.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A,A,M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36251,36251,36251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2319591,7CA2DABBCCDD33D1,684463372834134,20091209,20091209,5169,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.111492e+09,5.111492e+09,5.111492e+09,5.111492e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,181719461,181719461.0,181719461.0,181719461.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,94260,94360,94360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,0.0,10.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,40.0,40.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A,M,A,A

In [7]:
# check the logics behind R and confirm that the reprocessed line value should be added to the total
relevant_cols = (
    ["CLM_ID"]
    + [f"LINE_PRCSG_IND_CD_{i}" for i in range(1, 14)]
    + [f"LINE_NCH_PMT_AMT_{i}" for i in range(1, 14)]
    + [f"LINE_ALOWD_CHRG_AMT_{i}" for i in range(1, 14)]
    + [f"LINE_COINSRNC_AMT_{i}" for i in range(1, 14)]
)
has_R = carrier[relevant_cols].isin(["R"]).any(axis=1)
carrier_sample = carrier[has_R].sample(n=5, random_state=42)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
carrier_sample[relevant_cols].head()

,CLM_ID,LINE_PRCSG_IND_CD_1,LINE_PRCSG_IND_CD_2,LINE_PRCSG_IND_CD_3,LINE_PRCSG_IND_CD_4,LINE_PRCSG_IND_CD_5,LINE_PRCSG_IND_CD_6,LINE_PRCSG_IND_CD_7,LINE_PRCSG_IND_CD_8,LINE_PRCSG_IND_CD_9,LINE_PRCSG_IND_CD_10,LINE_PRCSG_IND_CD_11,LINE_PRCSG_IND_CD_12,LINE_PRCSG_IND_CD_13,LINE_NCH_PMT_AMT_1,LINE_NCH_PMT_AMT_2,LINE_NCH_PMT_AMT_3,LINE_NCH_PMT_AMT_4,LINE_NCH_PMT_AMT_5,LINE_NCH_PMT_AMT_6,LINE_NCH_PMT_AMT_7,LINE_NCH_PMT_AMT_8,LINE_NCH_PMT_AMT_9,LINE_NCH_PMT_AMT_10,LINE_NCH_PMT_AMT_11,LINE_NCH_PMT_AMT_12,LINE_NCH_PMT_AMT_13,LINE_ALOWD_CHRG_AMT_1,LINE_ALOWD_CHRG_AMT_2,LINE_ALOWD_CHRG_AMT_3,LINE_ALOWD_CHRG_AMT_4,LINE_ALOWD_CHRG_AMT_5,LINE_ALOWD_CHRG_AMT_6,LINE_ALOWD_CHRG_AMT_7,LINE_ALOWD_CHRG_AMT_8,LINE_ALOWD_CHRG_AMT_9,LINE_ALOWD_CHRG_AMT_10,LINE_ALOWD_CHRG_AMT_11,LINE_ALOWD_CHRG_AMT_12,LINE_ALOWD_CHRG_AMT_13,LINE_COINSRNC_AMT_1,LINE_COINSRNC_AMT_2,LINE_COINSRNC_AMT_3,LINE_COINSRNC_AMT_4,LINE_COINSRNC_AMT_5,LINE_COINSRNC_AMT_6,LINE_COINSRNC_AMT_7,LINE_COINSRNC_AMT_8,LINE_COINSRNC_AMT_9,LINE_COINSRNC_AMT_10,LINE_COINSRNC_AMT_11,LINE_COINSRNC_AMT_12,LINE_COINSRNC_AMT_13
2404099,684543373066765,A,A,A,A,R,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,160.0,60.0,160.0,170.0,190.0,170.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,200.0,70.0,210.0,220.0,210.0,240.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.0,30.0,40.0,50.0,50.0,40.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1879923,684173371257953,R,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2208685,684573373745860,R,A,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.0,20.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,20.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2160463,684113372998459,R,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1737497,684033372230316,A,A,R,A,A,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,90.0,190.0,0.0,0.0,60.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,110.0,120.0,0.0,0.0,90.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,10.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
# emergency fix to memory issues (leading to kernel crash)
amt_cols = [c for c in carrier.columns if any(
    c.startswith(p) for p in ['LINE_NCH_PMT_AMT', 'LINE_COINSRNC_AMT', 
                                'LINE_BENE_PTB_DDCT_AMT', 'LINE_BENE_PRMRY_PYR_PD_AMT',
                                'LINE_ALOWD_CHRG_AMT']
)]
carrier[amt_cols] = carrier[amt_cols].astype('float32')

ind_cols = [f"LINE_PRCSG_IND_CD_{i}" for i in range(1,14)]
carrier[ind_cols] = carrier[ind_cols].astype('category')

In [4]:
# prune columns
prefixes = ("PRF_PHYSN_NPI", "TAX_NUM", "LINE_ICD9_DGNS_CD")
prefix_columns = [c for c in carrier.columns if c.startswith(prefixes)]
carrier_cleaned = carrier.drop(columns=prefix_columns)
carrier_cleaned.head()

,DESYNPUF_ID,CLM_ID,CLM_FROM_DT,CLM_THRU_DT,ICD9_DGNS_CD_1,ICD9_DGNS_CD_2,ICD9_DGNS_CD_3,ICD9_DGNS_CD_4,ICD9_DGNS_CD_5,ICD9_DGNS_CD_6,...,LINE_PRCSG_IND_CD_4,LINE_PRCSG_IND_CD_5,LINE_PRCSG_IND_CD_6,LINE_PRCSG_IND_CD_7,LINE_PRCSG_IND_CD_8,LINE_PRCSG_IND_CD_9,LINE_PRCSG_IND_CD_10,LINE_PRCSG_IND_CD_11,LINE_PRCSG_IND_CD_12,LINE_PRCSG_IND_CD_13
0,000102649ED5601B,684813370207372,20080301,20080301,2720,49121,V5869,NaN,NaN,NaN,...,A,A,A,A,A,A,A,A,A,A
1,000102649ED5601B,684253371655461,20080407,20080407,1741,V4571,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,000102649ED5601B,684323369524620,20080420,20080420,V285,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,000102649ED5601B,684163371235030,20080524,20080524,7850,4011,78079,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,000102649ED5601B,684913369780400,20080604,20080604,33111,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
carrier1 = checking_claims_logic(carrier_cleaned)
carrier1.head()

,DESYNPUF_ID,CLM_ID,CLM_FROM_DT,CLM_THRU_DT,ICD9_DGNS_CD_1,ICD9_DGNS_CD_2,ICD9_DGNS_CD_3,ICD9_DGNS_CD_4,ICD9_DGNS_CD_5,ICD9_DGNS_CD_6,...,CARRIER_DDCTBL,CARRIER_PPPYMT,CARRIER_ALLOWED,CARRIER_BENERES,CARRIER_TOTAL_REIMB,CARRIER_TOTAL_PMT_PRE_REJECT,pct_rejected,has_msp_line,needs_review,reimb_valid
0,000102649ED5601B,684813370207372,20080301,20080301,2720,49121,V5869,NaN,NaN,NaN,...,0.0,0.0,40.0,0.0,40.0,40.0,0.0,False,False,True
1,000102649ED5601B,684253371655461,20080407,20080407,1741,V4571,NaN,NaN,NaN,NaN,...,0.0,0.0,110.0,10.0,50.0,50.0,0.0,True,False,True
2,000102649ED5601B,684323369524620,20080420,20080420,V285,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,30.0,10.0,70.0,70.0,0.0,False,False,False
3,000102649ED5601B,684163371235030,20080524,20080524,7850,4011,78079,NaN,NaN,NaN,...,0.0,0.0,240.0,50.0,110.0,110.0,0.0,False,False,True
4,000102649ED5601B,684913369780400,20080604,20080604,33111,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,80.0,10.0,50.0,50.0,0.0,False,False,True


In [7]:
carrier_sample = carrier1[carrier1["reimb_valid"] == False].sample(n=10, random_state=42)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
carrier_sample

,DESYNPUF_ID,CLM_ID,CLM_FROM_DT,CLM_THRU_DT,ICD9_DGNS_CD_1,ICD9_DGNS_CD_2,ICD9_DGNS_CD_3,ICD9_DGNS_CD_4,ICD9_DGNS_CD_5,ICD9_DGNS_CD_6,ICD9_DGNS_CD_7,ICD9_DGNS_CD_8,HCPCS_CD_1,HCPCS_CD_2,HCPCS_CD_3,HCPCS_CD_4,HCPCS_CD_5,HCPCS_CD_6,HCPCS_CD_7,HCPCS_CD_8,HCPCS_CD_9,HCPCS_CD_10,HCPCS_CD_11,HCPCS_CD_12,HCPCS_CD_13,LINE_NCH_PMT_AMT_1,LINE_NCH_PMT_AMT_2,LINE_NCH_PMT_AMT_3,LINE_NCH_PMT_AMT_4,LINE_NCH_PMT_AMT_5,LINE_NCH_PMT_AMT_6,LINE_NCH_PMT_AMT_7,LINE_NCH_PMT_AMT_8,LINE_NCH_PMT_AMT_9,LINE_NCH_PMT_AMT_10,LINE_NCH_PMT_AMT_11,LINE_NCH_PMT_AMT_12,LINE_NCH_PMT_AMT_13,LINE_BENE_PTB_DDCTBL_AMT_1,LINE_BENE_PTB_DDCTBL_AMT_2,LINE_BENE_PTB_DDCTBL_AMT_3,LINE_BENE_PTB_DDCTBL_AMT_4,LINE_BENE_PTB_DDCTBL_AMT_5,LINE_BENE_PTB_DDCTBL_AMT_6,LINE_BENE_PTB_DDCTBL_AMT_7,LINE_BENE_PTB_DDCTBL_AMT_8,LINE_BENE_PTB_DDCTBL_AMT_9,LINE_BENE_PTB_DDCTBL_AMT_10,LINE_BENE_PTB_DDCTBL_AMT_11,LINE_BENE_PTB_DDCTBL_AMT_12,LINE_BENE_PTB_DDCTBL_AMT_13,LINE_BENE_PRMRY_PYR_PD_AMT_1,LINE_BENE_PRMRY_PYR_PD_AMT_2,LINE_BENE_PRMRY_PYR_PD_AMT_3,LINE_BENE_PRMRY_PYR_PD_AMT_4,LINE_BENE_PRMRY_PYR_PD_AMT_5,LINE_BENE_PRMRY_PYR_PD_AMT_6,LINE_BENE_PRMRY_PYR_PD_AMT_7,LINE_BENE_PRMRY_PYR_PD_AMT_8,LINE_BENE_PRMRY_PYR_PD_AMT_9,LINE_BENE_PRMRY_PYR_PD_AMT_10,LINE_BENE_PRMRY_PYR_PD_AMT_11,LINE_BENE_PRMRY_PYR_PD_AMT_12,LINE_BENE_PRMRY_PYR_PD_AMT_13,LINE_COINSRNC_AMT_1,LINE_COINSRNC_AMT_2,LINE_COINSRNC_AMT_3,LINE_COINSRNC_AMT_4,LINE_COINSRNC_AMT_5,LINE_COINSRNC_AMT_6,LINE_COINSRNC_AMT_7,LINE_COINSRNC_AMT_8,LINE_COINSRNC_AMT_9,LINE_COINSRNC_AMT_10,LINE_COINSRNC_AMT_11,LINE_COINSRNC_AMT_12,LINE_COINSRNC_AMT_13,LINE_ALOWD_CHRG_AMT_1,LINE_ALOWD_CHRG_AMT_2,LINE_ALOWD_CHRG_AMT_3,LINE_ALOWD_CHRG_AMT_4,LINE_ALOWD_CHRG_AMT_5,LINE_ALOWD_CHRG_AMT_6,LINE_ALOWD_CHRG_AMT_7,LINE_ALOWD_CHRG_AMT_8,LINE_ALOWD_CHRG_AMT_9,LINE_ALOWD_CHRG_AMT_10,LINE_ALOWD_CHRG_AMT_11,LINE_ALOWD_CHRG_AMT_12,LINE_ALOWD_CHRG_AMT_13,LINE_PRCSG_IND_CD_1,LINE_PRCSG_IND_CD_2,LINE_PRCSG_IND_CD_3,LINE_PRCSG_IND_CD_4,LINE_PRCSG_IND_CD_5,LINE_PRCSG_IND_CD_6,LINE_PRCSG_IND_CD_7,LINE_PRCSG_IND_CD_8,LINE_PRCSG_IND_CD_9,LINE_PRCSG_IND_CD_10,LINE_PRCSG_IND_CD_11,LINE_PRCSG_IND_CD_12,LINE_PRCSG_IND_CD_13,CARRIER_REJECTED_AMT,CARRIER_NCHREIMB,CARRIER_COINS,CARRIER_DDCTBL,CARRIER_PPPYMT,CARRIER_ALLOWED,CARRIER_BENERES,CARRIER_TOTAL_REIMB,CARRIER_TOTAL_PMT_PRE_REJECT,pct_rejected,has_msp_line,needs_review,reimb_valid
4414843,EE64DC81E60E21E8,684203373396768,20080617,20080617,36552,37921,NaN,NaN,NaN,NaN,NaN,NaN,92083,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0,30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,50.0,10.0,60.0,0.0,90.0,70.0,120.0,120.0,0.0,False,False,False
414819,16457FB1838ABDD1,684953370629437,20081112,20081112,7948,78900,7906,7904,NaN,NaN,NaN,NaN,76700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,60.0,10.0,0.0,0.0,60.0,10.0,70.0,70.0,0.0,False,False,False
66213,03939D293781AE41,684073373373010,20080911,20080911,3940,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93307,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,130.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,120.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,100.0,0.0,130.0,0.0,120.0,130.0,230.0,230.0,0.0,False,False,False
1278877,4405823308D1A585,684293370227455,20081125,20081125,V7652,2190,NaN,NaN,NaN,NaN

In [16]:
carrier1["reimb_valid"].value_counts(normalize=True)

reimb_valid
True     0.687112
False    0.312888
Name: proportion, dtype: float64

In [12]:
# Given 31.29% claims failed the reimbursement logic check, confirmed the invalid rows had nothing to do with MSP as we intended
carrier1[~carrier1["reimb_valid"]]["has_msp_line"].value_counts()

has_msp_line
False    1481379
Name: count, dtype: int64

In [13]:
# for invalid rows, a majority of lines are completely normal accepted lines. This points strongly towards synthetic perturbation of data as the cause instead of business logics.
ind_cols = [f"LINE_PRCSG_IND_CD_{i}" for i in range(1,14)]
invalid_rows = carrier1[~carrier1['reimb_valid']]
invalid_rows[ind_cols].stack().value_counts()

A    2728144
O     219660
C     111593
M      31841
N       9847
R       6162
H       2813
B       2434
L       2204
G       1184
K        719
2        204
Z        105
J        101
1         16
=         14
0          3
E          3
F          2
Name: count, dtype: int64

In [15]:
# Medican overage is $30, 75th percentile is $70, maximum is $3910, and minimum is $10 (which rules out rounding as a possibility for invalidities). This again means the inconsistency isn't due to random noise. As a result, we should use beneficiary level information for costs as the predictor value.
carrier1['overage'] = carrier1['CARRIER_TOTAL_REIMB'] - carrier1['CARRIER_ALLOWED']
carrier1[~carrier1['reimb_valid']]['overage'].describe()

count    1.481379e+06
mean     6.271455e+01
std      1.005986e+02
min      1.000000e+01
25%      1.000000e+01
50%      3.000000e+01
75%      7.000000e+01
max      3.910000e+03
Name: overage, dtype: float64

In [19]:
# prune columns
prefixes = ("LINE_NCH_PMT_AMT", "LINE_BENE_PTB_DDCTBL_AMT", "LINE_BENE_PRMRY_PYR_PD_AMT", "LINE_COINSRNC_AMT", "LINE_ALOWD_CHRG_AMT", "LINE_PRCSG_IND_CD")
additional = ["CARRIER_NCHREIMB", "CARRIER_COINS", "CARRIER_DDCTBL", "CARRIER_PPPYMT", "CARRIER_ALLOWED", "CARRIER_BENERES", "CARRIER_TOTAL_REIMB", "CARRIER_TOTAL_PMT_PRE_REJECT", "has_msp_line",	"needs_review",	"reimb_valid"]
prefix_columns = [c for c in carrier1.columns if c.startswith(prefixes)]
carrier_cleaned = carrier1.drop(columns=prefix_columns + additional)
carrier_cleaned["overage"] = carrier_cleaned["overage"].clip(lower=0)
carrier_cleaned.head()

,DESYNPUF_ID,CLM_ID,CLM_FROM_DT,CLM_THRU_DT,ICD9_DGNS_CD_1,ICD9_DGNS_CD_2,ICD9_DGNS_CD_3,ICD9_DGNS_CD_4,ICD9_DGNS_CD_5,ICD9_DGNS_CD_6,ICD9_DGNS_CD_7,ICD9_DGNS_CD_8,HCPCS_CD_1,HCPCS_CD_2,HCPCS_CD_3,HCPCS_CD_4,HCPCS_CD_5,HCPCS_CD_6,HCPCS_CD_7,HCPCS_CD_8,HCPCS_CD_9,HCPCS_CD_10,HCPCS_CD_11,HCPCS_CD_12,HCPCS_CD_13,CARRIER_REJECTED_AMT,pct_rejected,overage
0,000102649ED5601B,684813370207372,20080301,20080301,2720,49121,V5869,NaN,NaN,NaN,NaN,NaN,71020,84450,83970,83970,82308,80051,84100,80061,83540,36415,84443,84100,84100,0.0,0.0,0.0
1,000102649ED5601B,684253371655461,20080407,20080407,1741,V4571,NaN,NaN,NaN,NaN,NaN,NaN,99213,76645,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
2,000102649ED5601B,684323369524620,20080420,20080420,V285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77057,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,40.0
3,000102649ED5601B,684163371235030,20080524,20080524,7850,4011,78079,NaN,NaN,NaN,NaN,NaN,93010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
4,000102649ED5601B,684913369780400,20080604,20080604,33111,NaN,NaN,NaN,NaN,NaN,NaN,NaN,99308,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0


In [ ]:
audit_df(inpatient, "inpatient", "DESYNPUF_ID")

In [ ]:
audit_df(outpatient, "outpatient", "DESYNPUF_ID")

In [ ]:
audit_df(pde, "pde", "DESYNPUF_ID")